<a href="https://colab.research.google.com/github/NielsRogge/Transformers-Tutorials/blob/master/DINO/Visualize_self_attention_of_DINO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 可视化DINO的自注意力

在本笔记本中，我们将可视化[DINO论文]的一些注意力模式(https://arxiv.org/abs/2104.14294). 该论文表明，当视觉变换器使用DINO方法以自我监督的方式进行预训练时，它们能够在没有明确训练的情况下分割图像中的对象。在ResNets等经典卷积模型中没有观察到这种行为。

在HuggingFace Transformers中，可以直接加载任何[DINO检查点](https://huggingface.co/models?other=dino)从集线器直接转换为“ViTModel”或“ViTForImageClassification”。

## 定义实用功能

在这里，我们定义了一些对可视化有用的函数。我们从[官方实施]中得到了这些(https://github.com/facebookresearch/dino/blob/main/visualize_attention.py).

In [ ]:
import skimage.io
from skimage.measure import find_contours
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

def apply_mask(image, mask, color, alpha=0.5):
    for c in range(3):
        image[:, :, c] = image[:, :, c] * (1 - alpha * mask) + alpha * mask * color[c] * 255
    return image

def random_colors(N, bright=True):
    """
    Generate random colors.
    """
    brightness = 1.0 if bright else 0.7
    hsv = [(i / N, 1, brightness) for i in range(N)]
    colors = list(map(lambda c: colorsys.hsv_to_rgb(*c), hsv))
    random.shuffle(colors)
    return colors

def display_instances(image, mask, fname="test", figsize=(5, 5), blur=False, contour=True, alpha=0.5):
    fig = plt.figure(figsize=figsize, frameon=False)
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)
    ax = plt.gca()

    N = 1
    mask = mask[None, :, :]
    # Generate random colors
    colors = random_colors(N)

    # Show area outside image boundaries.
    height, width = image.shape[:2]
    margin = 0
    ax.set_ylim(height + margin, -margin)
    ax.set_xlim(-margin, width + margin)
    ax.axis('off')
    masked_image = image.astype(np.uint32).copy()
    for i in range(N):
        color = colors[i]
        _mask = mask[i]
        if blur:
            _mask = cv2.blur(_mask,(10,10))
        # Mask
        masked_image = apply_mask(masked_image, _mask, color, alpha)
        # Mask Polygon
        # Pad to ensure proper polygons for masks that touch image edges.
        if contour:
            padded_mask = np.zeros((_mask.shape[0] + 2, _mask.shape[1] + 2))
            padded_mask[1:-1, 1:-1] = _mask
            contours = find_contours(padded_mask, 0.5)
            for verts in contours:
                # Subtract the padding and flip (y, x) to (x, y)
                verts = np.fliplr(verts) - 1
                p = Polygon(verts, facecolor="none", edgecolor=color)
                ax.add_patch(p)
    ax.imshow(masked_image.astype(np.uint8), aspect='auto')
    fig.savefig(fname)
    print(f"{fname} saved.")
    return

## 获取图片

在这里，我们使用我们熟悉的猫图像，这是COCO数据集的一部分。

In [ ]:
import requests
from PIL import Image

url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw).convert('RGB')
image

## 准备图像

接下来，我们使用ViTFeatureExtractor为模型准备图像（它将调整图像大小+规范化图像）。

In [ ]:
from mindnlp.transformers import ViTFeatureExtractor

feature_extractor = ViTFeatureExtractor.from_pretrained("facebook/dino-vits8", size=480)

In [ ]:
pixel_values = feature_extractor(images=image, return_tensors="ms").pixel_values 
print(pixel_values.shape)

## 前进传播

让我们通过模型来推进它。请注意，我们指定了`output_attentions=True `，因为我们需要注意力分数进行可视化。我们还指定`interpole_pos_encoding=True `，以确保预训练的位置嵌入被插值。这确保了该模型在与训练期间使用的图像分辨率不同的图像分辨率上工作。

In [ ]:
from mindnlp.transformers import ViTModel

model = ViTModel.from_pretrained("facebook/dino-vits8", add_pooling_layer=False)

In [ ]:
# forward pass
outputs = model(pixel_values, output_attentions=True, interpolate_pos_encoding=True)

## 可视化

最后，让我们可视化最后一层的注意力图！

In [ ]:
attentions = outputs.attentions[-1] # we are only interested in the attention maps of the last layer
nh = attentions.shape[1] # number of head

# we keep only the output patch attention
attentions = attentions[0, :, 0, 1:].reshape(nh, -1)
print(attentions.shape)

In [ ]:
import os

from mindnlp.core import nn, ops
#import torchvision

threshold = 0.6
w_featmap = pixel_values.shape[-2] // model.config.patch_size
h_featmap = pixel_values.shape[-1] // model.config.patch_size

# we keep only a certain percentage of the mass
val, idx = ops.sort(attentions)
val /= ops.sum(val, dim=1, keepdim=True)
cumval = ops.cumsum(val, dim=1)
th_attn = cumval > (1 - threshold)
idx2 = ops.argsort(idx)
for head in range(nh):
    th_attn[head] = th_attn[head][idx2[head]]
th_attn = th_attn.reshape(nh, w_featmap, h_featmap).float()
# interpolate
th_attn = nn.functional.interpolate(th_attn.unsqueeze(0), scale_factor=float(model.config.patch_size), mode="nearest", recompute_scale_factor=True)[0].asnumpy()

attentions = attentions.reshape(nh, w_featmap, h_featmap)
attentions = nn.functional.interpolate(attentions.unsqueeze(0), scale_factor=float(model.config.patch_size), mode="nearest", recompute_scale_factor=True)[0]
attentions = attentions.asnumpy()

# show and save attentions heatmaps
output_dir = '.'
os.makedirs(output_dir, exist_ok=True)

for j in range(nh):

    plt.figure()
    plt.imshow(attentions[j])
    